In [1]:
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = True

import numpy as np
import xarray as xr
import cupy as cp
import numba.cuda
import cupy_xarray

SIZE = (200, 1, 987, 1920)


U = xr.DataArray(
    name="U",
    data=np.random.random(SIZE).astype(np.float32),
    dims=["time", "face", "j", "i"],
)
V = xr.DataArray(
    name="V",
    data=np.random.random(SIZE).astype(np.float32),
    dims=["time", "face", "j", "i"],
)

ds = xr.merge([U, V])
ds_gpu = ds.copy().cupy.as_cupy()

In [2]:
ds

<xarray.Dataset> Size: 3GB
Dimensions:  (time: 200, face: 1, j: 987, i: 1920)
Dimensions without coordinates: time, face, j, i
Data variables:
    U        (time, face, j, i) float32 2GB 0.6393 0.9512 ... 0.9246 0.4779
    V        (time, face, j, i) float32 2GB 0.7954 0.3912 ... 0.6693 0.9309

In [9]:
%%time
u = ds["U"].data
v = ds["V"].data
uu = u * u
vv = v * v
uv = u * v
result = (uu.mean(), vv.mean(), uv.mean())

CPU times: user 1.49 s, sys: 1.23 s, total: 2.72 s
Wall time: 2.72 s


In [10]:
%%time
u = ds_gpu["U"].data
v = ds_gpu["V"].data
uu = u * u
vv = v * v
uv = u * v
result = (uu.mean(), vv.mean(), uv.mean())
cp.cuda.Stream.null.synchronize()

CPU times: user 7.07 ms, sys: 16.7 ms, total: 23.8 ms
Wall time: 21.7 ms


In [11]:
dtype = "float32"
with open("qm.cpp", "rt") as f:
    kernel_code = f.read()

U = ds_gpu["U"].data
V = ds_gpu["V"].data
    
module = cp.RawModule(code=kernel_code)
kernel = module.get_function("super_fast_kernel")

# Setup output array and call kernel
size = U.size

# Calculate grid and block dimensions for optimal occupancy
block_size = 512  # Must match the shared memory size in kernel
grid_size = min(4096, (size + block_size - 1) // block_size)

In [12]:
%%time
# Execute the kernel
results = cp.zeros(3, dtype=dtype)
kernel((grid_size,), (block_size,), (U, V, results, size))

# Compute means by dividing by size
results /= size

cp.cuda.Stream.null.synchronize()

CPU times: user 0 ns, sys: 6.51 ms, total: 6.51 ms
Wall time: 4.69 ms


In [13]:
from numba import cuda
import cupy

@cuda.jit
def numba_super_fast_kernel(u, v, results, size):
    # Shared memory for parallel reduction
    sdata = cuda.shared.array(shape=512, dtype=numba.float32)

    # Initialize accumulators for UU, VV, UV
    acc_uu = 0.0
    acc_vv = 0.0
    acc_uv = 0.0

    # Grid stride loop for processing large arrays
    for i in range(cuda.grid(1), size, cuda.gridDim.x * cuda.blockDim.x):
        u_val = u[i]
        v_val = v[i]

        # Compute products and accumulate in thread's registers
        acc_uu += u_val * u_val
        acc_vv += v_val * v_val
        acc_uv += u_val * v_val

    # First level of reduction in shared memory
    tid = cuda.threadIdx.x
    sdata[tid] = acc_uu
    cuda.syncthreads()

    # Parallel reduction for UU
    s = cuda.blockDim.x // 2
    while s > 0:
        if tid < s:
            sdata[tid] += sdata[tid + s]
        cuda.syncthreads()
        s //= 2

    # Thread 0 writes UU result using atomic add
    if tid == 0:
        cuda.atomic.add(results, 0, sdata[0])

    # Repeat for VV
    sdata[tid] = acc_vv
    cuda.syncthreads()

    s = cuda.blockDim.x // 2
    while s > 0:
        if tid < s:
            sdata[tid] += sdata[tid + s]
        cuda.syncthreads()
        s //= 2

    if tid == 0:
        cuda.atomic.add(results, 1, sdata[0])

    # Repeat for UV
    sdata[tid] = acc_uv
    cuda.syncthreads()

    s = cuda.blockDim.x // 2
    while s > 0:
        if tid < s:
            sdata[tid] += sdata[tid + s]
        cuda.syncthreads()
        s //= 2

    if tid == 0:
        cuda.atomic.add(results, 2, sdata[0])

def numba_super_fast(U: cupy.ndarray, V: cupy.ndarray) -> cupy.ndarray:
    """
    Numba implementation of the super_fast function that:
    1. Uses a single kernel for all operations
    2. Performs reduction in the same kernel to minimize memory traffic
    3. Uses Numba CUDA for high performance
    """

    # Setup the kernel
    size = U.size
    # results = cuda.device_array(3, dtype=U.dtype)
    # results.copy_to_device(np.zeros(3, dtype=U.dtype))
    results = cupy.zeros(3, dtype=U.dtype)

    # Calculate grid and block dimensions for optimal occupancy
    block_size = 512  # Must match the shared memory size in kernel
    grid_size = min(4096, (size + block_size - 1) // block_size)

    # Execute the kernel
    numba_super_fast_kernel[grid_size, block_size](U.ravel(), V.ravel(), results, size)
    results /= size

    # Synchronize to ensure completion
    cuda.synchronize()
    return results


In [14]:
%timeit _ = numba_super_fast(ds_gpu["U"].data, ds_gpu["V"].data)

4.98 ms ± 111 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
